## The State of Tax Justice: Requests for Journalists

- Author: Mario Cuenda García, based on Alison Schultz, based on Javier Garcia Bernado's work
- Created: 4 November 2024
- Last updated:

**Description**
- This notebook is the third out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
    - Details on the misalignment method and its background can be found here: https://www.sciencedirect.com/science/article/pii/S0305750X23003455. 
    - The working paper version is here: https://www.econstor.eu/bitstream/10419/286362/1/wp-2023-33.pdf 

- This notebook estimates profit misalignment based on different formulas. It uses the dataset **"data/final/cbcr_main.csv"** (for the estimation with imputed values) or the dataset **"data/final/cbcr_main_noimputation_allsubgroupsonly.csv"** (for the estimation without imputed values). 

**Outline**
1. Define misalignment
2. Calculate misalignment for sample with full information.
3. Calculate misalignment for samples with imputed data and aggregate results. 

**To dos before running this notebook**
- Run the notebooks 1_clean and 2_impute_missings. Note the requirements and instructions given in these notebooks.

**To dos in this notebook**

Change the formula to any required formula in each section and adapt the output name of the csvs. The formulas I have used are like follows, where sales refer to unrelated party revenues and assets to tangible assets excluding cash.

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]

- Adjust the input path in 5.1 to the bootstrapped sample you use

## 0. Load packages

In [1]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.0f}'.format

[TJN TOOLS: Data processing] Module loaded.
[TJN TOOLS: Other functions] Module loaded.
[TJN TOOLS: Paths] Module loaded. Sharepoint FOUND at /Users/mariocuendagarcia/Library/CloudStorage/OneDrive-SharedLibraries-TaxJusticeNetworkLtd


## Step 1. Generate the template datasets

### Step 1.1. Generate the dataset with Unique ISO parents

In [2]:
# Open the original dataset
iso_parents = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Keep just the following columns: iso_parent and year
iso_parents = iso_parents[['iso_parent', 'year']]

# Keep every unique combination of iso_parent and year
iso_parents = iso_parents.drop_duplicates(subset=['iso_parent', 'year'])

# Sort by year, then iso_parent
iso_parents = iso_parents.sort_values(by=['year', 'iso_parent'])

# Filter by year
iso_parents_2016= iso_parents[iso_parents['year'] == 2016]
iso_parents_2017= iso_parents[iso_parents['year'] == 2017]
iso_parents_2018= iso_parents[iso_parents['year'] == 2018]
iso_parents_2019= iso_parents[iso_parents['year'] == 2019]
iso_parents_2020= iso_parents[iso_parents['year'] == 2020]
iso_parents_2021= iso_parents[iso_parents['year'] == 2021]

# OPTIONAL: Print the count of how many unique iso_partner values there are
print(iso_parents_2016['iso_parent'].nunique())
print(iso_parents_2017['iso_parent'].nunique())
print(iso_parents_2018['iso_parent'].nunique())
print(iso_parents_2019['iso_parent'].nunique())
print(iso_parents_2020['iso_parent'].nunique())
print(iso_parents_2021['iso_parent'].nunique())

iso_parents_2021

26
38
46
50
52
52


,iso_parent,year
0,ARE,2021
155,ARG,2021
269,AUS,2021
765,AUT,2021
797,AZE,2021
839,BEL,2021
1008,BGR,2021
1028,BHR,2021
1068,BMU,2021
1649,BRA,2021


### Step 1.2. Generate the dataset with unique iso_partners

In [3]:
## Download the relevant dataset
iso_partners = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
iso_partners = iso_partners[~iso_partners['iso_partner'].isin(non_countries)]

# Keep just the following columns: iso_partner and year
iso_partners = iso_partners[['iso_partner', 'year']]

# Sort by year, then iso_partner
iso_partners = iso_partners.sort_values(by=['year', 'iso_partner'])

# Keep every unique combination of iso_partner and year
iso_partners = iso_partners.drop_duplicates(subset=['iso_partner', 'year'])

# Filter by year
iso_partners_2016= iso_partners[iso_partners['year'] == 2016]
iso_partners_2017= iso_partners[iso_partners['year'] == 2017]
iso_partners_2018= iso_partners[iso_partners['year'] == 2018]
iso_partners_2019= iso_partners[iso_partners['year'] == 2019]
iso_partners_2020= iso_partners[iso_partners['year'] == 2020]
iso_partners_2021= iso_partners[iso_partners['year'] == 2021]


# Optional: Print the count of how many unique iso_partner values there are 
print(iso_partners_2016['iso_partner'].nunique())
print(iso_partners_2017['iso_partner'].nunique())
print(iso_partners_2018['iso_partner'].nunique())
print(iso_partners_2019['iso_partner'].nunique())
print(iso_partners_2020['iso_partner'].nunique())
print(iso_partners_2021['iso_partner'].nunique())

183
215
213
210
212
211


### Step 1.3. Generate the template dataset

In [4]:
# Perform a cross join to merge all values of iso_partners_ with each value of iso_parents
iso_combinations_2016 = iso_parents_2016.assign(key=1).merge(iso_partners_2016.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2017 = iso_parents_2017.assign(key=1).merge(iso_partners_2017.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2018 = iso_parents_2018.assign(key=1).merge(iso_partners_2018.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2019 = iso_parents_2019.assign(key=1).merge(iso_partners_2019.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2020 = iso_parents_2020.assign(key=1).merge(iso_partners_2020.assign(key=1), on='key').drop('key', axis=1)
iso_combinations_2021 = iso_parents_2021.assign(key=1).merge(iso_partners_2021.assign(key=1), on='key').drop('key', axis=1)

# Concatenate all the years
template_dataset = pd.concat([iso_combinations_2016, iso_combinations_2017, iso_combinations_2018, iso_combinations_2019, iso_combinations_2020, iso_combinations_2021])

# Drop year_y
template_dataset = template_dataset.drop(columns=['year_y'])
# Rename year_x to year
template_dataset = template_dataset.rename(columns={'year_x': 'year'})
# Order columns by iso_parent then iso_partner then year
template_dataset = template_dataset[['iso_parent', 'iso_partner', 'year']]
# Generate new column called cbcr_estimates
template_dataset['cbcr_estimates'] = np.nan

template_dataset

,iso_parent,iso_partner,year,cbcr_estimates
0,AUS,ABW,2016,NaN
1,AUS,AFG,2016,NaN
2,AUS,AGO,2016,NaN
3,AUS,ALB,2016,NaN
4,AUS,AND,2016,NaN
...,...,...,...,...
10967,ZAF,XKV,2021,NaN
10968,ZAF,YEM,2021,NaN
10969,ZAF,ZAF,2021,NaN
10970,ZAF,ZMB,2021,NaN


## Step 2. Define the misalignment formula

In [5]:
def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",
                                         'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[.5, 0, 0, .5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15): 

    # Create variable with positive profits only for calculating shares
    cbcr_data['profit_var_pos'] = cbcr_data[profit_var]
    cbcr_data.loc[cbcr_data[profit_var] < 0, 'profit_var_pos'] = 0
    cbcr_data['share_profit'] = cbcr_data['profit_var_pos'] / cbcr_data.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f"share_{var}")
            actual_weights.append(weights[i])
            cbcr_data.loc[cbcr_data[var] < 0, var] = 0  # Set economic activity measure to zero if negative
            cbcr_data[f"share_{var}"] = cbcr_data[var] / cbcr_data.groupby('iso_parent')[var].transform('sum')

    # Calculate the share of economic activity
    cbcr_data["share_economy_partner_of_parent"] = (cbcr_data.loc[:, actual_variables] * actual_weights).sum(1, min_count=len(actual_weights))
    # Set economic activity to 1% for those jurisdictions without economic activity but with reported profits
    cbcr_data.loc[(cbcr_data["share_economy_partner_of_parent"] == 0) & (cbcr_data[profit_var] > 0), "share_economy_partner_of_parent"] = 0.01

    # Normalize the economic activity shares to sum to 1
    cbcr_data["share_economy_partner_of_parent"] = cbcr_data["share_economy_partner_of_parent"] / cbcr_data.groupby('iso_parent')["share_economy_partner_of_parent"].transform('sum')

    # Calculate theoretical profit and misaligned profit
    cbcr_data["theoretical_profit"] = cbcr_data["share_economy_partner_of_parent"] * cbcr_data.groupby('iso_parent')[profit_var].transform('sum')
    cbcr_data["misaligned_profit"] = cbcr_data[profit_var] - cbcr_data["theoretical_profit"]

    # Set positive misaligned profits to 0 if ETR exceeds the threshold (etr_max)
    cbcr_data.loc[((cbcr_data["misaligned_profit"] > 0) & (cbcr_data["etr_average_corrected"] > etr_max)), "misaligned_profit"] = 0

    # Adjust misalignment per 'iso_parent'
    def adjust_misalignment(group):
        total_negative_misalignment = group.loc[group["misaligned_profit"] < 0, "misaligned_profit"].sum()
        total_positive_misalignment = group.loc[group["misaligned_profit"] > 0, "misaligned_profit"].sum()
        
        # Adjust negative misalignments to balance positive misalignments within each 'iso_parent'
        if total_negative_misalignment != 0:
            factor = - total_positive_misalignment / total_negative_misalignment
            group.loc[group["misaligned_profit"] < 0, "misaligned_profit"] *= factor
        
        return group

    # Apply the adjustment by grouping by 'iso_parent'
    cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)

    return cbcr_data

## Step 3. Calculate misalignment for sample with full information

### Step 3.1 Import data
- Import data without imputed values. This is only the data from the sample of reporting countries that actually is reported on a country basis, i.e. excluding aggregated country groups and data from reporting countries that do not report on a country by country basis, but just by continents.

In [6]:
cbcr_sample = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Exclude country groups
cbcr_sample = cbcr_sample[~cbcr_sample['iso_partner'].isin(non_countries)]

### Step 3.2 Exclude countries that do not report truly country-by-country


- In the 2024 data, the following reporting countries do not report country-by-country. We exclude those from the "clean" analysis where we only use values that are actually in the data.
    - Austria: Only continents in all years
    - Czechia: Only Czechia versus rest of the world from 2019 to 2021
    - Finland: Only Finland and rest of the world between 2016 and 2018 and Finland and continents between 2019 and 2021
    - Greece: Only Greece and continents between 2017 and 2019
    - Hungary: Only Hungary versus rest of the world between 2018 and 2021
    - Isle of Man: Only continents between 2017 and 2020
    - Ireland: Only Ireland versus rest of the world in all years
    - Korea: Only Korea and rest of the world betweem 2016 and 2018 and Korea and continents between 2019 and 2021
    - Macau: Only Macau versus rest of the world between 2019 and 2021
    - Mauritius: Only Mauritius and continents between 2019 and 2021
    - Morocco: Only Morocco and continents in 2021
    - Netherlands: Only Netherlands versus rest of the world between 2016 and 2017
    - Norway: Only Norway and continents 2016 and 2017
    - New Zealand: Only New Zealand versus rest of the world between 2018 and 2021
    - Poland: Only Poland and continents 2019 to 2021
    - Sweden: Only Sweden and continents in all years
    - United Kingdom: Only UK and continents between 2017 and 2021

In [7]:
# Define the conditions for exclusion
exclusion_conditions = [
    ('AUT', 2016, 2021),                # Austria: all years
    ('CZE', 2019, 2021),                # Czechia: from 2019 to 2021
    ('FIN', 2016, 2021),                # Finland: all years
    ('GRC', 2017, 2019),                # Greece: between 2017 and 2019
    ('HUN', 2018, 2021),                # Hungary: between 2018 and 2021
    ('IMN', 2017, 2020),                # Isle of Man: between 2017 and 2020
    ('IRL', 2016, 2021),                # Ireland: all years
    ('KOR', 2016, 2021),                # Korea: all years
    ('MAC', 2019, 2021),                # Macau: between 2019 and 2021
    ('MUS', 2019, 2021),                # Mauritius: between 2019 and 2021
    ('MAR', 2021, 2021),                # Morocco: 2021
    ('NLD', 2016, 2017),                # Netherlands: between 2016 and 2017
    ('NOR', 2016, 2017),                # Norway: 2016 and 2017
    ('NZL', 2018, 2021),                # New Zealand: between 2018 and 2021
    ('POL', 2019, 2021),                # Poland: 2019 to 2021
    ('SWE', 2016, 2021),                # Sweden: all years
    ('GBR', 2017, 2021)                 # United Kingdom: between 2017 and 2021
]

# Iterate through the exclusion conditions
for iso_parent, start_year, end_year in exclusion_conditions:
    cbcr_sample = cbcr_sample[~((cbcr_sample['iso_parent'] == iso_parent) & 
                                     (cbcr_sample['year'].between(start_year, end_year)))]

### Step 3.3 Calculate misalignment for sample countries with full information

The order of the formula is as follow:

- formula_vars=['n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll", 'stated_capital' 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']

    - SOTJ: 50% employees, 50% payroll

        - weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0]

    - Canadian formula: 50% employees, 50% sales

        - weights=[1/2, 1/2, 0, 0, 0, 0, 0, 0]

    - CCCTB: 1/6 employees, 1/6 payroll, 1/3 sales, 1/3 assets

        - weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0]
        
    - Double-weighted sales: 1/4 employees, 1/2 sales, 1/4 assets

        - weights=[1/4, 1/2, 1/4, 0, 0, 0, 0, 0]

    - Three-factor: 1/3 employees, 1/3 sales, 1/3 assets

        - weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0]


The end of this cell shows, among other things, the misaligned profits and the theoretical profits, as well as the CBCR variables.

**More importantly, if for whatever reason we wanted to just run this cell without the "bad reporters", we could just modify the cell to run like the cell from Step 5.2. in order to obtain a dataset of profit shifting without the "bad reporters". In a way, this cell is not really necessary for the rest of the notebook. It just shows an intermediary steps if we want to calculate the misalignment for the "good reporters" only.**

In [ ]:
misalignment_2021 = cbcr_sample[cbcr_sample['year'] == 2021].copy()

# Calculate the misalignment for 2021
misalignment_2021 = calculate_misalignment(misalignment_2021, etr_max=0.15, weights=[1/2, 0, 0, 1/2, 0, 0, 0, 0])

# Keep only the first occurrence of these unique variables for each 'iso_partner'
unique_columns = misalignment_2021.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

# Keep iso_parent, iso_partner, year, misaligned_profit, theoretical_profit, profit_loss_before_income_tax_corrected, 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', and 'holding_or_managing_ip'
misalignment_2021 = misalignment_2021[['iso_parent', 'iso_partner', 'year', 'misaligned_profit', 'theoretical_profit', 'profit_loss_before_income_tax_corrected', 'n_employees', "unrelated_party_revenues", "tangible_assets_except_cash", "payroll",'stated_capital', 'total_revenues', 'related_party_revenues', 'holding_or_managing_ip']]

# Show if iso_partner = ESP and iso_parent = ESP
misalignment_2021[(misalignment_2021['iso_parent'] == 'ESP') | (misalignment_2021['iso_partner'] == 'ESP')]

/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_34944/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)


,iso_parent,iso_partner,year,misaligned_profit,theoretical_profit,profit_loss_before_income_tax_corrected,n_employees,unrelated_party_revenues,tangible_assets_except_cash,payroll,stated_capital,total_revenues,related_party_revenues,holding_or_managing_ip
1173,ESP,AGO,2021,"-4,837,982","37,851,280","24,224,782","1,820","99,657,046","6,187,906","3,136,355","4,263,218","103,783,928","4,126,883",0
1174,ESP,AND,2021,"-2,781,035","7,676,217","-156,752",157,"48,322,744","17,727,964","6,377,355","476,200,466","54,567,379","6,244,635",0
1175,ESP,ARE,2021,0,"107,381,546","412,060,799","2,287","944,968,290","1,912,190,965","86,755,426","13,124,576","1,791,676,435","846,708,145",1
1176,ESP,ARG,2021,"-91,121,936","1,972,005,906","1,715,354,938","81,236","16,108,900,162","18,402,713,893","531,106,995","7,313,039,059","17,637,367,648","1,528,467,486",3
1177,ESP,AUS,2021,"-278,974,097","1,291,930,273","506,181,251","22,273","10,120,023,664","3,662,965,491","1,185,681,773","11,172,666,318","10,532,761,568","412,737,904",6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1273,ESP,URY,2021,"137,936,848","252,056,502","389,993,350","9,030","2,414,243,548","806,344,723","104,519,657","2,727,480,263","3,354,314,763","940,071,215",1
1274,ESP,USA,2021,0,"6,524,975,193","10,527,379,930","109,801","79,067,644,471","88,909,492,659","6,061,182,537","74,509,895,692","91,002,422,172","11,934,777,701",43
1275,ESP,VEN,2021,0,"346,214,392","463,358,996","16,211","3,868,942,637","2,727,138,013","40,489,564","3,137,607,927","4,203,619,709","334,677,072",0
1276,ESP,VNM,2021,"-13,983,807","5,762,233","-33,624,086",261,"88,956,330","33,470,307","912,330","5,924,974","94,176,423","5,220,093",0


In [9]:
cbcr_eu = cbcr_sample[cbcr_sample['eu'] == 1]

results_eu = []

for year in range(first_year, first_year + n_years):
    print(f"Total profit shifted in USD mn {year} within the EU")
    
    misalignment_year_eu = cbcr_eu[cbcr_eu['year'] == year].copy()
    misalignment_year_eu = calculate_misalignment(misalignment_year_eu, etr_max=0.15, weights=[1/3, 1/3, 1/3, 0, 0, 0, 0, 0])

    # Keep only the first occurrence of these unique variables for each 'iso_partner'
    unique_columns_eu = misalignment_year_eu.drop_duplicates(subset=['iso_partner'])[['iso_partner', 'partner_jurisdiction', 
                                                                               'etr_average_corrected', 'cit',
                                                                               'tax_revenue_current_usd', 
                                                                               'gvt_health_expenditure', 'region_tjn', 
                                                                               'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

    # Perform the groupby operation on 'iso_partner'
    country_results_year_eu = misalignment_year_eu.groupby(['iso_partner']).agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()

    # Convert results to millions
    country_results_year_eu['negative_misalignment'] = -country_results_year_eu['negative_misalignment'] / 1e6
    country_results_year_eu['positive_misalignment'] = country_results_year_eu['positive_misalignment'] / 1e6
    country_results_year_eu['theoretical_profit'] = country_results_year_eu['theoretical_profit'] / 1e6
    country_results_year_eu['reported_profit'] = country_results_year_eu['reported_profit'] / 1e6

    # Merge the unique columns back into the result
    country_results_year_eu = country_results_year_eu.merge(unique_columns_eu, on='iso_partner', how='left')

    # Calculate other relevant variables
    country_results_year_eu['tax_revenue_loss'] = country_results_year_eu['negative_misalignment'] * country_results_year_eu['cit']
    country_results_year_eu['tax_revenue_gain'] = country_results_year_eu['positive_misalignment'] * country_results_year_eu['etr_average_corrected']

    country_results_year_eu['tax_revenue_loss_pct_of_gvt_health_expenditure'] = np.where(
        country_results_year_eu['gvt_health_expenditure'] == 0, 
        np.nan, 
        country_results_year_eu['tax_revenue_loss'] / (country_results_year_eu['gvt_health_expenditure'] / 1e6)
    )
    
    country_results_year_eu['tax_revenue_loss_pct_of_total_tax_revenues'] = np.where(
        country_results_year_eu['tax_revenue_current_usd'] == 0, 
        np.nan, 
        country_results_year_eu['tax_revenue_loss'] / (country_results_year_eu['tax_revenue_current_usd'] / 1e6)
    )

    # Calculate totals
    total_positive_misalignment_eu = country_results_year_eu['positive_misalignment'].sum()
    total_negative_misalignment_eu = country_results_year_eu['negative_misalignment'].sum()
    total_tax_revenue_loss_eu = country_results_year_eu['tax_revenue_loss'].sum()
    total_tax_revenue_gain_eu = country_results_year_eu['tax_revenue_gain'].sum()
    average_tax_revenue_loss_pct_of_gvt_health_expenditure_eu = country_results_year_eu['tax_revenue_loss_pct_of_gvt_health_expenditure'].mean()
    average_tax_revenue_loss_pct_of_total_tax_revenues_eu = country_results_year_eu['tax_revenue_loss_pct_of_total_tax_revenues'].mean()

    print(f"EU results: Year {year}: Positive Misalignment: {total_positive_misalignment_eu}, Negative Misalignment: {total_negative_misalignment_eu}, "
          f"Total tax revenue loss: {total_tax_revenue_loss_eu}, Total tax revenue gain: {total_tax_revenue_gain_eu}")

    # Calculate countries' fractions of totals
    country_results_year_eu['tax_revenue_loss_caused_pct_of_total'] = country_results_year_eu['positive_misalignment'] / total_positive_misalignment_eu
    country_results_year_eu['tax_revenue_loss_caused_usd'] = country_results_year_eu['tax_revenue_loss_caused_pct_of_total'] * total_tax_revenue_loss_eu
    country_results_year_eu['tax_revenue_loss_suffered_pct_of_total'] = country_results_year_eu['tax_revenue_loss'] / total_tax_revenue_loss_eu

    country_results_year_eu = country_results_year_eu[['iso_partner', 'partner_jurisdiction', 'negative_misalignment',
       'tax_revenue_loss', 'tax_revenue_loss_suffered_pct_of_total', 'tax_revenue_loss_pct_of_gvt_health_expenditure',
       'tax_revenue_loss_pct_of_total_tax_revenues', 'positive_misalignment', 'tax_revenue_gain', 
       'tax_revenue_loss_caused_usd', 'tax_revenue_loss_caused_pct_of_total', 'etr_average_corrected', 'cit',
       'region_tjn', 'ukt', 'oecd', 'oecd_oct', 'nld_oct']]

    country_results_year_eu = country_results_year_eu.sort_values(by='iso_partner')
    country_results_year_eu.to_csv(f'{output_tables}/EU_analyses/EU_misalignment_onethirdeach_{year}.csv', index=False)
    
    # Append aggregate results to the list
    results_eu.append({
        'year': year,
        'total_positive_misalignment': total_positive_misalignment_eu,
        'total_negative_misalignment': total_negative_misalignment_eu,
        'total_tax_revenue_loss': total_tax_revenue_loss_eu,
        'total_tax_revenue_gain': total_tax_revenue_gain_eu,
        'average_tax_revenue_loss_pct_of_gvt_health_expenditure': average_tax_revenue_loss_pct_of_gvt_health_expenditure_eu,
        'average_tax_revenue_loss_pct_of_total_tax_revenues': average_tax_revenue_loss_pct_of_total_tax_revenues_eu
    })

# Convert aggregate results to a DataFrame
results_eu_df = pd.DataFrame(results_eu)

# Save the aggregated results to a CSV or Excel file
results_eu_df.to_csv(f'{output_tables}/EU_analyses/EU_misalignment_onethirdeach_formula.csv', index=False)

Total profit shifted in USD mn 2016 within the EU


/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_34944/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_34944/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby

EU results: Year 2016: Positive Misalignment: 150068.17730629534, Negative Misalignment: 150068.1773062953, Total tax revenue loss: 42722.34653517235, Total tax revenue gain: 12762.66531671885
Total profit shifted in USD mn 2017 within the EU
EU results: Year 2017: Positive Misalignment: 283522.2533652137, Negative Misalignment: 283522.2533652137, Total tax revenue loss: 78371.1784294765, Total tax revenue gain: 24195.090116563533
Total profit shifted in USD mn 2018 within the EU
EU results: Year 2018: Positive Misalignment: 263375.7355634665, Negative Misalignment: 263375.73556346644, Total tax revenue loss: 71059.86061596162, Total tax revenue gain: 22430.883771710935
Total profit shifted in USD mn 2019 within the EU
EU results: Year 2019: Positive Misalignment: 377002.65527275123, Negative Misalignment: 377002.65527275123, Total tax revenue loss: 102150.6792058986, Total tax revenue gain: 34757.894257681975
Total profit shifted in USD mn 2020 within the EU
EU results: Year 2020: Pos

/var/folders/pm/bp4z4lln39xcwn73chrtp3n00000gn/T/ipykernel_34944/421381151.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cbcr_data = cbcr_data.groupby('iso_parent', group_keys=False).apply(lambda x: adjust_misalignment(x)).reset_index(drop=True)
